# Módulo 13: Interpretabilidad, equidad y ciclo de vida

[Abrir en Colab](https://colab.research.google.com/github/sgevatschnaider/data-science-business-decisions/blob/main/notebooks/13-ia-responsable.ipynb)

**Pregunta de decisión:** ¿Podemos justificar, controlar y monitorear una decisión algorítmica frente a personas afectadas y responsables del negocio?

**Autor:** Sergio Gevatschnaider


## Objetivos

- Distinguir explicación global, local y causal.
- Medir desempeño y errores por grupos relevantes.
- Documentar datos, modelo, usos previstos y límites.
- Diseñar monitoreo de calidad, drift, valor y daño.

**Criterio de éxito:** el resultado debe cambiar o sostener una acción concreta, superar una referencia y declarar límites.


## 1. Entorno reproducible

Registramos versiones y semilla antes de producir evidencia. Ejecutá siempre **Runtime → Run all** en Colab.


In [ ]:
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

SEED = 42
np.random.seed(SEED)
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})

## 2. Experimento base

El bloque siguiente construye una referencia mínima y verificable. No representa todavía la recomendación final.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(42)
tabla = pd.DataFrame({
    "grupo": np.repeat(["A", "B"], 300),
    "real": np.r_[rng.binomial(1, 0.30, 300), rng.binomial(1, 0.45, 300)],
})
tabla["score"] = np.clip(0.15 + 0.62*tabla["real"] + rng.normal(0, 0.22, 600), 0, 1)

def metricas(grupo, umbral=0.5):
    subset = tabla.query("grupo == @grupo")
    pred = (subset["score"] >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(subset["real"], pred, labels=[0, 1]).ravel()
    return {"grupo": grupo, "seleccion": pred.mean(), "fpr": fp/(fp+tn), "fnr": fn/(fn+tp)}

pd.DataFrame([metricas("A"), metricas("B")])

## 3. Evidencia visual

Una visualización útil permite comparar, muestra unidades y deja visible la incertidumbre o variación relevante.


In [ ]:
import matplotlib.pyplot as plt

group_metrics = pd.DataFrame([metricas("A"), metricas("B")]).set_index("grupo")
ax = group_metrics[["fpr", "fnr"]].plot.bar(figsize=(8, 3.5), color=["#1d4ed8", "#be123c"])
ax.set(ylabel="Tasa", title="Errores por grupo")
ax.set_ylim(0, 1)
plt.tight_layout()

## 4. Comparación para decidir

Un modelo prioriza solicitudes para revisión. La organización debe medir errores por grupo, explicar casos, registrar cambios y definir apelación.

La tabla fuerza una comparación entre alternativas, costos o criterios. Adaptala a las unidades del caso.


In [ ]:
pd.concat([pd.DataFrame([metricas('A', u), metricas('B', u)]).assign(umbral=u) for u in [.4, .5, .6]], ignore_index=True)

## 5. Desafío de transferencia

**Construí un registro de riesgos con dueño, indicador, umbral, respuesta y evidencia de cierre.**

1. Identificar partes afectadas y daños plausibles.
2. Comparar métricas globales y segmentadas.
3. Generar una explicación global y una local con límites.
4. Diseñar indicadores, alertas y responsable de respuesta.

Antes de continuar, escribí una hipótesis, una condición que la refutaría y el costo de una decisión equivocada.

### Registro de decisión

Completá la celda siguiente como evidencia de cierre del laboratorio.


In [ ]:
decision_record = {
    "pregunta": '¿Podemos justificar, controlar y monitorear una decisión algorítmica frente a personas afectadas y responsables del negocio?',
    "hipotesis": "Completar antes del análisis",
    "evidencia": "Registrar la tabla o visualización que cambia la decisión",
    "recomendacion": "Expresar acción, población y horizonte",
    "limitacion": "Indicar qué podría invalidar la conclusión",
    "responsable": "Asignar dueño y fecha de revisión",
}
pd.Series(decision_record, name="registro_de_decision")

## 6. Cierre verificable

**Entregable:** Ficha de modelo con propósito, métricas globales y por grupo, explicación, riesgos, monitoreo y protocolo de intervención humana.

- Hallazgo principal:
- Evidencia que lo respalda:
- Comparación contra baseline o escenario alternativo:
- Limitación:
- Acción, responsable y fecha de revisión:

Material elaborado por el profesor Sergio Gevatschnaider.
